In [ ]:
from datasets import load_dataset

#spider_train = load_dataset("xlangai/spider", split="train")

spider_val = load_dataset("xlangai/spider", split="validation")

In [ ]:
from src.data_loaders.spider_data import get_spider_schema_ddl_and_candidates


spider_schema_ddls_and_candidates = get_spider_schema_ddl_and_candidates()

In [ ]:
for q in spider_val:
    schema = spider_schema_ddls_and_candidates[q['db_id']]
    schema_string = "\n".join([table_info['ddl'] for table_info in schema])
    print(schema_string)



In [ ]:

schema = spider_schema_ddls_and_candidates[spider_val[0]['db_id']]
schema_string = "\n".join([table_info['ddl'] for table_info in schema])
print(schema_string)


In [ ]:
candidate_list = []

for table_info in schema:
    candidates = table_info['candidates']
    table_name = candidates['table']
    columns = candidates['columns']
    for column in columns:
        candidate_list.append(f"{table_name}.{column}")

print(candidate_list)

In [ ]:
schema = spider_schema_ddls_and_candidates[q['db_id']]
current_db = ""

for q in spider_val:
    if current_db != q['db_id']:
        schema = spider_schema_ddls_and_candidates[q['db_id']]
        current_db = q['db_id']
        candidate_list = [
        f"{t['candidates']['table']}.{col}"
        for t in schema
        for col in t['candidates']['columns']
        ]
        print(candidate_list)

In [2]:
from types import SimpleNamespace


# Regeln einmal im State statt als Criteria pro Kandidat - spart ~90% der Request-Tokens
RULES = ("A column counts as referenced if it appears in SELECT, WHERE, a JOIN condition, GROUP BY, "
         "HAVING, ORDER BY, or inside an aggregation function of the SQL query that answers the question. "
         "If a table is used only via * or COUNT(*) and no other column of it is referenced, "
         "its first column counts as referenced.")


def merge_responses(responses):
    # Fuehrt die Responses aller Chunks einer Frage zu einem Objekt mit .nouls und .usage zusammen.
    # Jede Tabelle landet in genau einem Chunk, daher ueberschneiden sich die nouls nicht.
    return SimpleNamespace(
        nouls={k: a for resp in responses for k, a in resp.nouls.items()},
        usage=SimpleNamespace(
            input_tokens=sum(resp.usage.input_tokens for resp in responses),
            output_tokens=sum(resp.usage.output_tokens for resp in responses),
        ),
    )


def run_jev_schema_linking(data):
    results = []

    from typesafe_sdk import Noul, TypeSafeClient


    results = []

    with TypeSafeClient(model="jev-latest") as client:
        for q in data:
            responses = []                
            for input in q['input']:
                schema_string = input['ddl']
                candidate_list = input['candidate_strings']
                gold_schema = q['gold_schema']
                try:
                    response = client.system_one(
                        state={"question": q['question'], "schema": schema_string, "rules": RULES},
                        questions={c: Noul(instructions=f"Is column {c} referenced in the SQL query?")
                                    for c in candidate_list},
                    )
                    responses.append(response)
                except Exception as e:
                    print(f"Error processing question {q['question']} with db_id {q['db']}: {e}")
            results.append({"question": q['question'], "db_id": q['db'], "sql": q['sql'], "response": merge_responses(responses), "gold_schema": gold_schema})
    return results


In [2]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

data = get_spider_ent_data(max_tokens=6000)

for q in data:
    print(len(q['input']))

35
35
35
35
35
35
34
34
35
34
35
35
34
34
34
35
35
35
35
35
34
35
35
35
35
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
28
31
31
32
32
32
32
31
32
32
32
31
32
32
31
31
32
32
32
32
32
32
32
31
32
32
32
31
32
32
32
31
31
31
32
32
32
32
32
32
32
32
32
31
32
31
31
32
32
32
32
32
31
32
32
32
31
31
31
32
31
31
31
31
31
31
31
32
32
31
32
32
31
32
31
37
37
38
37
37
37
37
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
33
33
33
33
33
33
33
33
33
33
33
33
33
33
33
33
34
35
35
35
35
35
35
35
35
35
35
35
35
35
35
35
35
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
37
30
30
30
30
30
30
30
30
30
30
30
30
30
33
33
33
33
33
33
33
33
33
33
34
33
33
33
33
33
33
33
33
34
34
33
33
33
33
33
35
35
34
35
34
35
35
35
35
35
35
35
35
35
34
35
35
34
34
35
35
35
35
35
35
35
35
35
34
35
35
34
35
35
34
35
35
35
35
35
35
35
3

In [4]:
from src.data_loaders.spider_ent_data import get_spider_ent_data


results = run_jev_schema_linking(get_spider_ent_data(max_tokens=6000))


Error processing question What is the total number of countries where Spanish is spoken by the largest percentage of people? with db_id geography_environment_and_climate: POST https://api.typesafe.ai/v1/systemone: 499 Unable to complete this request. (request_id=req_01a13addf7e17f24b7b4deb96b68dc20)
Error processing question What is the name of the conductor who has worked the greatest number of years? with db_id arts_culture_and_media: POST https://api.typesafe.ai/v1/systemone: 499 Unable to complete this request. (request_id=req_01a13ae11cf37c61be0552b702ca782c)


In [3]:
import json
def response_to_dict(resp):
    return {
        "nouls": {k: a.noul for k, a in resp.nouls.items()},
        "usage": {"input_tokens": resp.usage.input_tokens,
                  "output_tokens": resp.usage.output_tokens},
    }

def save_results(results, file_name):
    serializable = []
    for r in results:
        item = {**r, "response": response_to_dict(r["response"])}
        # Nur bei run_jev_two_stage vorhanden: zusammengefuehrte Responses aus Stufe 1 (Tabellen)
        if "table_response" in r:
            item["table_response"] = response_to_dict(r["table_response"])
        serializable.append(item)
    with open(f"{file_name}.json", "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=4)

In [5]:
# Eval der Jev-Results: micro P/R/F6 über (table, column)-Paare, wie in src/QLora/evaluate.py

def _gold_cols(gold):
    return {(t.lower(), c.lower()) for t, cols in gold.items() for c in (cols or []) if t and c}

def _scores(response):
    # 'table.column' -> noul-Wahrscheinlichkeit
    return {tuple(k.lower().split(".", 1)): a.noul for k, a in response.nouls.items()}

def eval_jev(results, threshold=0.5, beta=6):
    tp = n_pred = n_gold = 0
    perfect_recall = 0
    for r in results:
        gold = _gold_cols(r["gold_schema"])
        pred = {k for k, p in _scores(r["response"]).items() if p >= threshold}
        tp += len(pred & gold)
        n_pred += len(pred)
        n_gold += len(gold)
        perfect_recall += gold <= pred
    p = tp / n_pred if n_pred else 0.0
    rec = tp / n_gold if n_gold else 0.0
    b2 = beta ** 2
    f = (1 + b2) * p * rec / (b2 * p + rec) if (b2 * p + rec) else 0.0
    return {"threshold": threshold, "precision": p, "recall": rec, f"f{beta}": f,
            "perfect_recall_rate": perfect_recall / len(results),
            "avg_pred_cols": n_pred / len(results), "tp": tp, "pred": n_pred, "gold": n_gold}

def present_results(results):
    print(f"Fragen: {len(results)}")

    # Gold-Spalten, die gar nicht unter den Kandidaten sind (sollte 0 sein)
    missing = sum(len(_gold_cols(r["gold_schema"]) - set(_scores(r["response"]))) for r in results)
    print(f"Gold-Spalten ohne Kandidat: {missing}")

    usage_in = sum(r["response"].usage.input_tokens for r in results)
    usage_out = sum(r["response"].usage.output_tokens for r in results)
    print(f"Tokens: input={usage_in:,}  output={usage_out:,}\n")

    print(f"{'thr':>5} {'P':>7} {'R':>7} {'F6':>7} {'R=1':>7} {'#pred':>7}")
    for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        m = eval_jev(results, thr)
        print(f"{thr:>5.2f} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f6']:>7.4f} "
            f"{m['perfect_recall_rate']:>7.4f} {m['avg_pred_cols']:>7.2f}")

    # Fehleranalyse bei 0.5: Fragen mit verpassten Gold-Spalten
    misses = []
    for r in results:
        s = _scores(r["response"])
        missed = {k: s.get(k) for k in _gold_cols(r["gold_schema"]) if s.get(k, 0) < 0.5}
        if missed:
            misses.append((r["question"], r["sql"], missed))
    print(f"\nFragen mit verpassten Gold-Spalten @0.5: {len(misses)}")
    for qtext, sql, missed in misses[:10]:
        print(f"- {qtext}\n  {sql}\n  missed: {missed}")



In [6]:
# Zweistufiges Schema Linking mit Jev:
#   Stufe 1: kompaktes Schema (Tabellen, Spalten, FK-Ziele) -> ein Noul pro Tabelle
#   Stufe 2: volle DDL nur der ausgewaehlten Tabellen -> ein Noul pro Spalte (wie run_jev_schema_linking)
import random

from typesafe_sdk import Noul, TypeSafeClient

from src.utils import _extract_fk_targets, _pack_tables_into_chunks

TABLE_RULES = ("A table is needed if at least one of its columns is referenced in the SQL query that answers the question, "
               "or if it is needed to join other needed tables. "
               "The schema shown is only one part of a large database; the needed tables may be in another part.")


def compact_table(t):
    # 'tabelle(spalte1, spalte2) FK -> ziel1, ziel2' - ohne Typen, Backticks und Beispielwerte
    line = f"{t['candidates']['table']}({', '.join(t['candidates']['columns'])})"
    fks = _extract_fk_targets(t['ddl'])
    if fks:
        line += f" FK -> {', '.join(sorted(fks))}"
    return line


def chunk_compact(tables, max_chars):
    # Greedy wie _pack_tables_into_chunks, aber nach Zeichen auf dem kompakten Format
    chunks, current, size = [], [], 0
    for t in tables:
        line = compact_table(t)
        if current and size + len(line) > max_chars:
            chunks.append(current)
            current, size = [], 0
        current.append((t['candidates']['table'], line))
        size += len(line) + 1
    if current:
        chunks.append(current)
    return chunks


def fk_neighbors(tables, selected):
    # Tabellen, die per FK direkt mit einer ausgewaehlten Tabelle verbunden sind (beide Richtungen)
    neighbors = set()
    for t in tables:
        name = t['candidates']['table'].lower()
        targets = {x.lower() for x in _extract_fk_targets(t['ddl'])}
        if name in selected:
            neighbors |= targets
        if targets & selected:
            neighbors.add(name)
    return neighbors - selected


def _ask(client, q, state, questions, responses):
    try:
        responses.append(client.system_one(state=state, questions=questions))
    except Exception as e:
        print(f"Error processing question {q['question']} with db_id {q['db']}: {e}")


def run_jev_two_stage(data, table_threshold=0.3, include_fk_neighbors=False,
                      stage1_max_chars=12000, stage2_max_tokens=6000):
    results = []
    with TypeSafeClient(model="jev-latest") as client:
        for q in data:
            # Stufe 1: Tabellen
            table_responses = []
            for chunk in chunk_compact(q['db_schema'], stage1_max_chars):
                _ask(client, q,
                     state={"question": q['question'], "schema": "\n".join(line for _, line in chunk), "rules": TABLE_RULES},
                     questions={name: Noul(instructions=f"Is table {name} needed for the SQL query?") for name, _ in chunk},
                     responses=table_responses)
            table_scores = {k: a.noul for resp in table_responses for k, a in resp.nouls.items()}

            selected = {t.lower() for t, p in table_scores.items() if p >= table_threshold}
            if include_fk_neighbors:
                selected |= fk_neighbors(q['db_schema'], selected)
            selected_tables = [t for t in q['db_schema'] if t['candidates']['table'].lower() in selected]

            # Stufe 2: Spalten der ausgewaehlten Tabellen
            column_responses = []
            for chunk in _pack_tables_into_chunks(selected_tables, q['question'], stage2_max_tokens, q['db']):
                _ask(client, q,
                     state={"question": q['question'], "schema": chunk['ddl'], "rules": RULES},
                     questions={c: Noul(instructions=f"Is column {c} referenced in the SQL query?") for c in chunk['candidate_strings']},
                     responses=column_responses)

            results.append({"question": q['question'], "db_id": q['db'], "sql": q['sql'], "gold_schema": q['gold_schema'],
                            "table_scores": table_scores, "selected_tables": sorted(selected),
                            "table_response": merge_responses(table_responses),
                            "response": merge_responses(column_responses)})
    return results


def stage1_table_recall(results, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    # Recall auf Tabellenebene: Wie viele Gold-Tabellen haette Stufe 1 bei welcher Schwelle durchgelassen?
    print(f"{'thr':>5} {'R_tab':>7} {'R=1':>7} {'#tab':>7}")
    for thr in thresholds:
        hit = total = n_sel = all_found = 0
        for r in results:
            gold = {t.lower() for t in r['gold_schema'] if t}
            sel = {t.lower() for t, p in r['table_scores'].items() if p >= thr}
            hit += len(gold & sel)
            total += len(gold)
            n_sel += len(sel)
            all_found += gold <= sel
        print(f"{thr:>5.2f} {hit / total:>7.4f} {all_found / len(results):>7.4f} {n_sel / len(results):>7.2f}")


In [9]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

# 30 Fragen zufaellig ueber alle Spider-Ent-DBs (die ersten 30 waeren alle aus arts_culture_and_media)
spider_ent = get_spider_ent_data(max_tokens=6000)
sample = random.Random(42).sample(spider_ent, 30)

results_two_stage = run_jev_two_stage(sample, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)

# Vergleich mit dem einstufigen Lauf auf denselben 30 Fragen (falls 'results' noch im Kernel ist)
sample_keys = {(q['question'], q['sql']) for q in sample}
print("\nEinstufig, dieselben Fragen:")
present_results([r for r in results if (r['question'], r['sql']) in sample_keys])


Stufe 1 (Tabellen):
  thr   R_tab     R=1    #tab
 0.10  1.0000  1.0000  139.20
 0.20  1.0000  1.0000   57.23
 0.30  1.0000  1.0000   36.00
 0.40  1.0000  1.0000   24.70
 0.50  1.0000  1.0000   15.97
Tokens Stufe 1: input=1,114,627

Stufe 2 (Spalten):
Fragen: 30
Gold-Spalten ohne Kandidat: 0
Tokens: input=553,279  output=351,417

  thr       P       R      F6     R=1   #pred
 0.10  0.0131  1.0000  0.3302  1.0000  207.90
 0.20  0.0362  0.9878  0.5772  0.9667   74.67
 0.30  0.0652  0.9878  0.7146  0.9667   41.40
 0.40  0.1001  0.9634  0.7813  0.9000   26.30
 0.50  0.1529  0.9024  0.7969  0.7333   16.13
 0.60  0.2292  0.8049  0.7537  0.6333    9.60
 0.70  0.4359  0.6220  0.6149  0.4333    3.90
 0.80  0.6275  0.3902  0.3943  0.3000    1.70
 0.90  0.0000  0.0000  0.0000  0.0000    0.00

Fragen mit verpassten Gold-Spalten @0.5: 8
- Which airline has abbreviation 'UAL'?
  SELECT Airline FROM AIRLINES WHERE Abbreviation  =  "UAL"
  missed: {('xelion21_flycore_air_carrier', 'airline_name'): 0.3

In [ ]:
save_results(results_two_stage, "spider_ent_jev_two_stage_results_30_samples_seed_42")

In [7]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

spider_ent = get_spider_ent_data(max_tokens=6000)

results_two_stage = run_jev_two_stage(spider_ent, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)



Stufe 1 (Tabellen):
  thr   R_tab     R=1    #tab
 0.10  1.0000  1.0000  136.17
 0.20  1.0000  1.0000   56.87
 0.30  0.9966  0.9950   32.68
 0.40  0.9966  0.9950   21.57
 0.50  0.9840  0.9767   14.89
Tokens Stufe 1: input=22,511,036

Stufe 2 (Spalten):
Fragen: 602
Gold-Spalten ohne Kandidat: 5
Tokens: input=10,057,031  output=6,451,895

  thr       P       R      F6     R=1   #pred
 0.10  0.0278  0.9951  0.5131  0.9900   96.21
 0.20  0.0767  0.9815  0.7443  0.9568   34.42
 0.30  0.1382  0.9506  0.8203  0.8920   18.51
 0.40  0.2112  0.9154  0.8397  0.8289   11.67
 0.50  0.3082  0.8586  0.8191  0.7508    7.50
 0.60  0.4345  0.7160  0.7037  0.5864    4.44
 0.70  0.5853  0.4574  0.4601  0.3355    2.10
 0.80  0.7853  0.1784  0.1822  0.1096    0.61
 0.90  0.8750  0.0173  0.0178  0.0249    0.05

Fragen mit verpassten Gold-Spalten @0.5: 150
- What are the names and release years for all the songs of the youngest singer?
  SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
 